In [ ]:
from pathlib import Path
from collections import defaultdict
import copy,hashlib,json,random,time,shutil,math
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from torchvision import models,transforms as T
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score,confusion_matrix
SEED=20261018
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(2); torch.backends.cudnn.benchmark=True
assert torch.cuda.is_available(),'Enable GPU before running this cell'
DEVICE=torch.device('cuda:0'); OUT=Path('/kaggle/working'); OUT.mkdir(exist_ok=True)
ROOT=Path('/kaggle/input/competitions/2026task2')
if not ROOT.exists(): ROOT=Path('/kaggle/input/2026task2')
OLD=Path('/kaggle/input/notebooks/shabiba2026/2026task2-fer-deep-learning')
assert (OLD/'fer_resnet18_best.pt').exists()
print('OPTIMIZATION_START',str(ROOT),str(OLD),torch.cuda.get_device_name(0),flush=True)
LABELS={'angry':0,'fear':1,'happy':2,'sad':3,'surprise':4,'neutral':5}
template=pd.read_csv(ROOT/'submission.csv'); assert template.columns.tolist()==['ID','Emotion'] and template.ID.is_unique
from concurrent.futures import ThreadPoolExecutor
reference=pd.read_csv(OLD/'fer_training_index.csv')
paths=reference.path.tolist(); Y=reference.label.to_numpy(dtype=int)
assert len(paths)==28275 and np.array_equal(Y,np.array([LABELS[(ROOT/p).parent.name.lower()] for p in paths]))
def read_image3(path):
 with Image.open(path) as im: return np.array(im.convert('L'))
with ThreadPoolExecutor(max_workers=16) as pool:
 arrays=list(pool.map(read_image3,[ROOT/p for p in paths]))
print('TRAIN_IMAGES_CACHED',len(arrays),flush=True)
with ThreadPoolExecutor(max_workers=16) as pool:
 test_arrays=list(pool.map(read_image3,[ROOT/'fer_data/fer_data/test'/str(name) for name in template.ID]))
labels=Y.tolist(); groups=[hashlib.sha256(str(image.shape).encode()+image.tobytes()).hexdigest() for image in arrays]
assert groups==reference.group.tolist()
va=np.load(OLD/'fer_valid_indices.npy'); tr=np.setdiff1d(np.arange(len(Y)),va)
assert not set(np.array(groups)[tr]) & set(np.array(groups)[va])
print('DATA_READY',len(tr),len(va),len(test_arrays),flush=True)
norm=T.Normalize([.485,.456,.406],[.229,.224,.225])
SIZE3=160; EPOCHS3=30; BATCH3=96
train_tf3=T.Compose([T.Resize((SIZE3,SIZE3)),T.RandomHorizontalFlip(),T.RandomAffine(8,translate=(.04,.04),scale=(.92,1.08)),T.ColorJitter(brightness=.15,contrast=.15),T.ToTensor(),norm,T.RandomErasing(p=.15,scale=(.01,.05))])
def eval_transform(size): return T.Compose([T.Resize((size,size)),T.ToTensor(),norm])
class Faces3(Dataset):
 def __init__(self,images,targets,tf): self.images,self.targets,self.tf=images,targets,tf
 def __len__(self): return len(self.images)
 def __getitem__(self,i): return self.tf(Image.fromarray(self.images[i]).convert('RGB')),int(self.targets[i])
def dl3(images,targets,tf,train=False): return DataLoader(Faces3(images,targets,tf),batch_size=BATCH3,shuffle=train,num_workers=2,pin_memory=True)
train_dl3=dl3([arrays[i] for i in tr],Y[tr],train_tf3,True)
valid_dl3=dl3([arrays[i] for i in va],Y[va],eval_transform(SIZE3))
test_dl3=dl3(test_arrays,np.zeros(len(test_arrays)),eval_transform(SIZE3))
@torch.inference_mode()
def pred3(net,dl):
 net.eval(); result=[]
 for x,_ in dl:
  x=x.to(DEVICE,non_blocking=True)
  with torch.autocast('cuda'): prob=(net(x).softmax(1)+net(x.flip(-1)).softmax(1))/2
  result.append(prob.float().cpu().numpy())
 return np.concatenate(result)
net3=models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
net3.fc=nn.Sequential(nn.Dropout(.3),nn.Linear(net3.fc.in_features,6)); net3=net3.to(DEVICE)
ema3=copy.deepcopy(net3).eval()
for p in ema3.parameters(): p.requires_grad_(False)
optimizer3=torch.optim.AdamW([{'params':[p for n,p in net3.named_parameters() if not n.startswith('fc.')],'lr':1e-4},{'params':net3.fc.parameters(),'lr':5e-4}],weight_decay=1e-4)
scheduler3=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer3,EPOCHS3,eta_min=1e-6)
criterion3=nn.CrossEntropyLoss(label_smoothing=.03); scaler3=torch.amp.GradScaler('cuda')
@torch.no_grad()
def update3(avg,net,decay=.99):
 for a,b in zip(avg.parameters(),net.parameters()): a.mul_(decay).add_(b,alpha=1-decay)
 for a,b in zip(avg.buffers(),net.buffers()): a.copy_(b)
def train_epoch3(net,avg,dl,opt):
 net.train(); total=0; loss_sum=0.; correct=0.
 for x,y in dl:
  x,y=x.to(DEVICE,non_blocking=True),y.to(DEVICE,non_blocking=True); lam=1.; perm=torch.arange(len(y),device=DEVICE)
  if random.random()<.35:
   lam=float(np.random.beta(.15,.15)); perm=torch.randperm(len(y),device=DEVICE); x=lam*x+(1-lam)*x[perm]
  opt.zero_grad(set_to_none=True)
  with torch.autocast('cuda'): z=net(x); loss=lam*criterion3(z,y)+(1-lam)*criterion3(z,y[perm])
  scaler3.scale(loss).backward(); scaler3.unscale_(opt); nn.utils.clip_grad_norm_(net.parameters(),5.); scaler3.step(opt); scaler3.update(); update3(avg,net)
  loss_sum+=loss.item()*len(y); total+=len(y); correct+=lam*(z.argmax(1)==y).sum().item()+(1-lam)*(z.argmax(1)==y[perm]).sum().item()
 return loss_sum/total,correct/total
hist3=[]; best3=-1.; stale3=0; start3=time.time()
for e in range(EPOCHS3):
 loss,acc=train_epoch3(net3,ema3,train_dl3,optimizer3); scheduler3.step(); pv=pred3(ema3,valid_dl3); score=float(accuracy_score(Y[va],pv.argmax(1)))
 row={'epoch':e+1,'train_loss':loss,'train_mixup_accuracy':acc,'validation_accuracy':score,'seconds':time.time()-start3}; hist3.append(row); pd.DataFrame(hist3).to_csv(OUT/'fer_resnet50_history.csv',index=False); print('RESNET50_EPOCH',json.dumps(row),flush=True)
 if score>best3:
  best3=score; best_ep3=e+1; stale3=0; state3={k:v.detach().cpu().clone() for k,v in ema3.state_dict().items()}
  torch.save({'state_dict':state3,'model':'resnet50_dropout','image_size':SIZE3,'epoch':best_ep3,'validation_accuracy':best3,'label_map':LABELS},OUT/'fer_resnet50_best.pt'); np.save(OUT/'fer_resnet50_valid_probs.npy',pv)
 else: stale3+=1
 if stale3>=8 and e>=19: break
ema3.load_state_dict(state3)
new_valid=np.load(OUT/'fer_resnet50_valid_probs.npy')
base_valid=(np.load(OLD/'fer_valid_probs.npy')+np.load(OLD/'fer_effnet_valid_192.npy'))/2
weights3=[0.,.25,.5,.75,1.]; ensemble_scores3={str(w):float(accuracy_score(Y[va],((1-w)*base_valid+w*new_valid).argmax(1))) for w in weights3}; chosen_w3=max(weights3,key=lambda w:ensemble_scores3[str(w)])
partial_test3=pred3(ema3,test_dl3); np.save(OUT/'fer_resnet50_partial_test_probs.npy',partial_test3)
print('RESNET50_VALIDATION_FINISHED',json.dumps({'best_epoch':best_ep3,'best_accuracy':best3,'ensemble_scores':ensemble_scores3,'chosen_weight':chosen_w3}),flush=True)
full3=copy.deepcopy(ema3).to(DEVICE)
for p in full3.parameters(): p.requires_grad_(True)
full_avg3=copy.deepcopy(full3).eval()
for p in full_avg3.parameters(): p.requires_grad_(False)
all_dl3=dl3(arrays,Y,train_tf3,True); all_opt3=torch.optim.AdamW(full3.parameters(),lr=2e-5,weight_decay=1e-4); all_sch3=torch.optim.lr_scheduler.CosineAnnealingLR(all_opt3,6,eta_min=1e-6)
full_hist3=[]
for e in range(6):
 loss,acc=train_epoch3(full3,full_avg3,all_dl3,all_opt3); all_sch3.step(); row={'epoch':e+1,'loss':loss,'train_mixup_accuracy':acc}; full_hist3.append(row); pd.DataFrame(full_hist3).to_csv(OUT/'fer_resnet50_full_history.csv',index=False); print('RESNET50_FULL_EPOCH',json.dumps(row),flush=True)
torch.save({'state_dict':full_avg3.state_dict(),'model':'resnet50_dropout','image_size':SIZE3,'epochs':6,'official_training_images':len(Y),'label_map':LABELS},OUT/'fer_resnet50_full.pt')
new_full_test=pred3(full_avg3,test_dl3); np.save(OUT/'fer_resnet50_full_test_probs.npy',new_full_test)
old_eff=models.efficientnet_b2(weights=None); old_eff.classifier=nn.Sequential(nn.Dropout(.3),nn.Linear(old_eff.classifier[1].in_features,6)); old_eff.load_state_dict(torch.load(OLD/'fer_effnet_b2_full.pt',map_location='cpu',weights_only=True)['state_dict']); old_eff=old_eff.to(DEVICE)
old_eff_test=pred3(old_eff,dl3(test_arrays,np.zeros(len(test_arrays)),eval_transform(192))); np.save(OUT/'fer_effnet_full_test_192.npy',old_eff_test)
old_full_test=(np.load(OLD/'fer_resnet18_full_test_probs.npy')+old_eff_test)/2
assert np.array_equal(old_full_test.argmax(1),pd.read_csv(OLD/'submission_fer_inference.csv').Emotion.to_numpy())
known_labels=defaultdict(set)
for image,label,g in zip(arrays,Y,groups):
 if float(np.std(image))>1.: known_labels[g].add(int(label))
def save_submission3(probs,name,consistent=True):
 frame=template.copy(); frame.Emotion=probs.argmax(1).astype(int); changes=0
 if consistent:
  for i,image in enumerate(test_arrays):
   if float(np.std(image))<=1.: continue
   g=hashlib.sha256(str(image.shape).encode()+image.tobytes()).hexdigest(); known=known_labels.get(g,set())
   if len(known)==1:
    label=next(iter(known)); changes+=int(frame.at[i,'Emotion']!=label); frame.at[i,'Emotion']=label
 assert frame.ID.equals(template.ID) and frame.Emotion.between(0,5).all() and not frame.isna().any().any()
 file=OUT/name; frame.to_csv(file,index=False)
 return {'filename':name,'rows':len(frame),'sha256':hashlib.sha256(file.read_bytes()).hexdigest(),'consistency_changes':changes}
records3=[]
records3.append(save_submission3(new_full_test,'submission_fer_resnet50_full.csv'))
records3.append(save_submission3((1-chosen_w3)*old_full_test+chosen_w3*new_full_test,'submission_fer_three_model_selected.csv'))
records3.append(save_submission3(.5*old_full_test+.5*new_full_test,'submission_fer_three_model_half.csv'))
report3={'date':'2026-10-18','architecture':'ResNet50 ImageNet1K V2','seed':SEED,'training_data':'official competition training labels only; ImageNet pretraining; no external FER or test labels','image_size':SIZE3,'best_epoch':best_ep3,'epochs_trained':len(hist3),'validation_split':'same pixel-grouped 22600/5675 split as previous run','new_validation_accuracy':best3,'ensemble_validation_scores':ensemble_scores3,'selected_new_weight':chosen_w3,'full_training_epochs':6,'full_model_validation_accuracy':None,'confusion_matrix':confusion_matrix(Y[va],new_valid.argmax(1)).tolist(),'submissions':records3,'seconds':time.time()-start3}
(OUT/'fer_optimization_20261018.json').write_text(json.dumps(report3,indent=2))
print('OPTIMIZATION_FINISHED',json.dumps(report3),flush=True)


In [ ]:
assert 'report3' in globals(),'Run the new training cell first'
valid_res3={160:new_valid}; options4=[]
for size in [128,192,224]:
 valid_res3[size]=pred3(ema3,dl3([arrays[i] for i in va],Y[va],eval_transform(size)))
 np.save(OUT/('fer_resnet50_valid_'+str(size)+'.npy'),valid_res3[size]); print('RESNET50_RESOLUTION',size,float(accuracy_score(Y[va],valid_res3[size].argmax(1))),flush=True)
for size,prob in valid_res3.items():
 for w in weights3:
  score=float(accuracy_score(Y[va],((1-w)*base_valid+w*prob).argmax(1)))
  options4.append({'resolution':size,'resnet50_weight':w,'validation_accuracy':score})
choice4=max(options4,key=lambda r:r['validation_accuracy']); size=choice4['resolution']; w=choice4['resnet50_weight']
valid_ensemble4=(1-w)*base_valid+w*valid_res3[size]
if size==160: test_res4=new_full_test
else: test_res4=pred3(full_avg3,dl3(test_arrays,np.zeros(len(test_arrays)),eval_transform(size)))
np.save(OUT/('fer_resnet50_full_test_'+str(size)+'.npy'),test_res4)
test_ensemble4=(1-w)*old_full_test+w*test_res4
prior4=np.bincount(Y,minlength=6)/len(Y); pred_freq4=(np.bincount(valid_ensemble4.argmax(1),minlength=6)+1)/(len(va)+6); factors4=prior4/pred_freq4
calibration4={str(g):float(accuracy_score(Y[va],(valid_ensemble4*factors4**g).argmax(1))) for g in [0.,.25,.5,.75,1.]}
gamma4=max([0.,.25,.5,.75,1.],key=lambda g:calibration4[str(g)])
files4=[save_submission3(test_ensemble4,'submission_fer_resnet50_tta_ensemble.csv'),save_submission3(test_ensemble4*factors4**gamma4,'submission_fer_resnet50_calibrated.csv')]
record4={'date':'2026-10-18','selection_data':'untouched pre-refit grouped validation models; no test labels','resolution_ensemble_options':options4,'selected_resolution_and_weight':choice4,'class_frequency_factors':factors4.tolist(),'calibration_validation_scores':calibration4,'selected_gamma':gamma4,'submissions':files4}
(OUT/'fer_refinement_20261018.json').write_text(json.dumps(record4,indent=2)); print('REFINEMENT_FINISHED',json.dumps(record4),flush=True)


In [ ]:
assert 'record4' in globals(),'Finish the ResNet50 inference comparison first'
import cv2
cv2.setNumThreads(1)
clahe_c=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(4,4))
images_c=[clahe_c.apply(a) for a in arrays]; tests_c=[clahe_c.apply(a) for a in test_arrays]
seed_c=20261019; random.seed(seed_c); np.random.seed(seed_c); torch.manual_seed(seed_c); torch.cuda.manual_seed_all(seed_c)
size_c=128; epochs_c=30
transform_c=T.Compose([T.Resize((size_c,size_c)),T.RandomHorizontalFlip(),T.RandomAffine(8,translate=(.05,.05),scale=(.9,1.05)),T.ColorJitter(brightness=.1,contrast=.1),T.ToTensor(),norm,T.RandomErasing(p=.1,scale=(.01,.04))])
train_c=dl3([images_c[i] for i in tr],Y[tr],transform_c,True); valid_c=dl3([images_c[i] for i in va],Y[va],eval_transform(size_c)); test_c=dl3(tests_c,np.zeros(len(tests_c)),eval_transform(size_c))
net_c=models.resnet34(weights=models.ResNet34_Weights.DEFAULT); net_c.fc=nn.Sequential(nn.Dropout(.2),nn.Linear(net_c.fc.in_features,6)); net_c=net_c.to(DEVICE)
avg_c=copy.deepcopy(net_c).eval()
for p in avg_c.parameters(): p.requires_grad_(False)
optimizer_c=torch.optim.AdamW([{'params':[p for n,p in net_c.named_parameters() if not n.startswith('fc.')],'lr':2e-4},{'params':net_c.fc.parameters(),'lr':7e-4}],weight_decay=1e-4)
schedule_c=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_c,epochs_c,eta_min=1e-6)
original_criterion3=criterion3
class_weights_c=(np.bincount(Y[tr],minlength=6).mean()/np.bincount(Y[tr],minlength=6))**.25
criterion3=nn.CrossEntropyLoss(weight=torch.tensor(class_weights_c,dtype=torch.float32,device=DEVICE),label_smoothing=.03)
best_c=-1.; stale_c=0; history_c=[]; start_c=time.time()
for e in range(epochs_c):
 loss,acc=train_epoch3(net_c,avg_c,train_c,optimizer_c); schedule_c.step(); probs_c=pred3(avg_c,valid_c); score_c=float(accuracy_score(Y[va],probs_c.argmax(1)))
 row_c={'epoch':e+1,'loss':loss,'train_mixup_accuracy':acc,'validation_accuracy':score_c,'seconds':time.time()-start_c}; history_c.append(row_c); pd.DataFrame(history_c).to_csv(OUT/'fer_clahe_resnet34_history.csv',index=False); print('CLAHE_RESNET34_EPOCH',json.dumps(row_c),flush=True)
 if score_c>best_c:
  best_c=score_c; best_ep_c=e+1; stale_c=0; state_c={k:v.detach().cpu().clone() for k,v in avg_c.state_dict().items()}
  torch.save({'state_dict':state_c,'model':'resnet34_dropout','image_size':size_c,'preprocess':'CLAHE clip2 grid4','epoch':best_ep_c,'validation_accuracy':best_c,'label_map':LABELS},OUT/'fer_clahe_resnet34_best.pt'); np.save(OUT/'fer_clahe_resnet34_valid.npy',probs_c)
 else: stale_c+=1
 if stale_c>=8 and e>=19: break
avg_c.load_state_dict(state_c); valid_probs_c=np.load(OUT/'fer_clahe_resnet34_valid.npy')
weights_c=[0.,.25,.5,.75,1.]; scores_c={str(w):float(accuracy_score(Y[va],((1-w)*valid_ensemble4+w*valid_probs_c).argmax(1))) for w in weights_c}; chosen_c=max(weights_c,key=lambda w:scores_c[str(w)])
print('CLAHE_SELECTION',json.dumps({'single_validation':best_c,'ensemble_scores':scores_c,'chosen_weight':chosen_c}),flush=True)
full_c=copy.deepcopy(avg_c).to(DEVICE)
for p in full_c.parameters(): p.requires_grad_(True)
full_avg_c=copy.deepcopy(full_c).eval()
for p in full_avg_c.parameters(): p.requires_grad_(False)
full_dl_c=dl3(images_c,Y,transform_c,True); full_opt_c=torch.optim.AdamW(full_c.parameters(),lr=2e-5,weight_decay=1e-4); full_sch_c=torch.optim.lr_scheduler.CosineAnnealingLR(full_opt_c,5,eta_min=1e-6); history_full_c=[]
for e in range(5):
 loss,acc=train_epoch3(full_c,full_avg_c,full_dl_c,full_opt_c); full_sch_c.step(); row_c={'epoch':e+1,'loss':loss,'train_mixup_accuracy':acc}; history_full_c.append(row_c); pd.DataFrame(history_full_c).to_csv(OUT/'fer_clahe_resnet34_full_history.csv',index=False); print('CLAHE_RESNET34_FULL',json.dumps(row_c),flush=True)
torch.save({'state_dict':full_avg_c.state_dict(),'model':'resnet34_dropout','image_size':size_c,'preprocess':'CLAHE clip2 grid4','official_training_images':len(Y),'epochs':5,'label_map':LABELS},OUT/'fer_clahe_resnet34_full.pt')
test_probs_c=pred3(full_avg_c,test_c); np.save(OUT/'fer_clahe_resnet34_full_test.npy',test_probs_c)
ensemble_v_c=(1-chosen_c)*valid_ensemble4+chosen_c*valid_probs_c; ensemble_t_c=(1-chosen_c)*test_ensemble4+chosen_c*test_probs_c
factors_c=(np.bincount(Y,minlength=6)/len(Y))/((np.bincount(ensemble_v_c.argmax(1),minlength=6)+1)/(len(va)+6))
scores_cal_c={str(g):float(accuracy_score(Y[va],(ensemble_v_c*factors_c**g).argmax(1))) for g in [0.,.25,.5,.75,1.]}; gamma_c=max([0.,.25,.5,.75,1.],key=lambda g:scores_cal_c[str(g)])
files_c=[save_submission3(test_probs_c,'submission_fer_clahe_resnet34.csv'),save_submission3(ensemble_t_c,'submission_fer_four_model_selected.csv'),save_submission3(ensemble_t_c*factors_c**gamma_c,'submission_fer_four_model_calibrated.csv')]
report_c={'date':'2026-10-18','model':'ResNet34 ImageNet pretraining + CLAHE','size':size_c,'seed':seed_c,'best_epoch':best_ep_c,'epochs_trained':len(history_c),'single_validation_accuracy':best_c,'validation_split':'unchanged original pixel-grouped split','class_loss_weights':class_weights_c.tolist(),'ensemble_validation_scores':scores_c,'selected_clahe_weight':chosen_c,'calibration_validation_scores':scores_cal_c,'selected_gamma':gamma_c,'full_training_epochs':5,'full_model_validation_accuracy':None,'training_data':'official train labels and ImageNet only; no external FER/test labels','submissions':files_c,'seconds':time.time()-start_c}
(OUT/'fer_complement_20261018.json').write_text(json.dumps(report_c,indent=2)); criterion3=original_criterion3
print('COMPLEMENT_FINISHED',json.dumps(report_c),flush=True)


In [ ]:
assert 'report_c' in globals(),'Finish the CLAHE comparison first'
import sys,subprocess,importlib.util,gc
if importlib.util.find_spec('facenet_pytorch') is None:
 subprocess.check_call([sys.executable,'-m','pip','install','--quiet','--no-deps','facenet-pytorch==2.6.0'],timeout=180)
from facenet_pytorch import InceptionResnetV1
for name in ['net3','ema3','full3','full_avg3','old_eff','net_c','avg_c','full_c','full_avg_c']:
 obj=globals().get(name)
 if isinstance(obj,nn.Module): obj.cpu()
for name in ['optimizer3','all_opt3','optimizer_c','full_opt_c']: globals().pop(name,None)
gc.collect(); torch.cuda.empty_cache()
seed_f=20261020; random.seed(seed_f); np.random.seed(seed_f); torch.manual_seed(seed_f); torch.cuda.manual_seed_all(seed_f)
size_f=160; epochs_f=26; norm_f=T.Normalize([127.5/255]*3,[128./255]*3)
train_tf_f=T.Compose([T.Resize((size_f,size_f)),T.RandomHorizontalFlip(),T.RandomAffine(8,translate=(.04,.04),scale=(.92,1.06)),T.ColorJitter(brightness=.1,contrast=.1),T.ToTensor(),norm_f,T.RandomErasing(p=.1,scale=(.01,.04))])
eval_tf_f=T.Compose([T.Resize((size_f,size_f)),T.ToTensor(),norm_f])
train_f=dl3([arrays[i] for i in tr],Y[tr],train_tf_f,True); valid_f=dl3([arrays[i] for i in va],Y[va],eval_tf_f); test_f=dl3(test_arrays,np.zeros(len(test_arrays)),eval_tf_f)
net_f=InceptionResnetV1(pretrained='vggface2',classify=True,num_classes=6,dropout_prob=.4).to(DEVICE)
avg_f=copy.deepcopy(net_f).eval()
for p in avg_f.parameters(): p.requires_grad_(False)
optimizer_f=torch.optim.AdamW([{'params':[p for n,p in net_f.named_parameters() if not n.startswith('logits.')],'lr':5e-5},{'params':net_f.logits.parameters(),'lr':5e-4}],weight_decay=1e-4)
scheduler_f=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_f,epochs_f,eta_min=1e-6)
criterion3=nn.CrossEntropyLoss(label_smoothing=.03)
best_f=-1.; stale_f=0; history_f=[]; start_f=time.time()
for e in range(epochs_f):
 loss,acc=train_epoch3(net_f,avg_f,train_f,optimizer_f); scheduler_f.step(); v_f=pred3(avg_f,valid_f); score_f=float(accuracy_score(Y[va],v_f.argmax(1)))
 row_f={'epoch':e+1,'loss':loss,'train_mixup_accuracy':acc,'validation_accuracy':score_f,'seconds':time.time()-start_f}; history_f.append(row_f); pd.DataFrame(history_f).to_csv(OUT/'fer_facenet_history.csv',index=False); print('FACENET_EPOCH',json.dumps(row_f),flush=True)
 if score_f>best_f:
  best_f=score_f; best_ep_f=e+1; stale_f=0; state_f={k:v.detach().cpu().clone() for k,v in avg_f.state_dict().items()}
  torch.save({'state_dict':state_f,'model':'InceptionResnetV1','pretraining':'VGGFace2 identity','size':160,'epoch':best_ep_f,'validation_accuracy':best_f,'label_map':LABELS},OUT/'fer_facenet_best.pt'); np.save(OUT/'fer_facenet_valid.npy',v_f)
 else: stale_f+=1
 if stale_f>=7 and e>=17: break
avg_f.load_state_dict(state_f); valid_probs_f=np.load(OUT/'fer_facenet_valid.npy')
weights_f=[0.,.25,.5,.75,1.]; scores_f={str(w):float(accuracy_score(Y[va],((1-w)*ensemble_v_c+w*valid_probs_f).argmax(1))) for w in weights_f}; chosen_f=max(weights_f,key=lambda w:scores_f[str(w)])
print('FACENET_SELECTION',json.dumps({'single_validation':best_f,'ensemble_scores':scores_f,'chosen_weight':chosen_f}),flush=True)
full_f=copy.deepcopy(avg_f).to(DEVICE)
for p in full_f.parameters(): p.requires_grad_(True)
full_avg_f=copy.deepcopy(full_f).eval()
for p in full_avg_f.parameters(): p.requires_grad_(False)
all_f=dl3(arrays,Y,train_tf_f,True); full_opt_f=torch.optim.AdamW(full_f.parameters(),lr=1e-5,weight_decay=1e-4); full_sch_f=torch.optim.lr_scheduler.CosineAnnealingLR(full_opt_f,5,eta_min=1e-6); history_full_f=[]
for e in range(5):
 loss,acc=train_epoch3(full_f,full_avg_f,all_f,full_opt_f); full_sch_f.step(); row_f={'epoch':e+1,'loss':loss,'train_mixup_accuracy':acc}; history_full_f.append(row_f); pd.DataFrame(history_full_f).to_csv(OUT/'fer_facenet_full_history.csv',index=False); print('FACENET_FULL',json.dumps(row_f),flush=True)
torch.save({'state_dict':full_avg_f.state_dict(),'model':'InceptionResnetV1','pretraining':'VGGFace2 identity','size':160,'epochs':5,'official_training_images':len(Y),'label_map':LABELS},OUT/'fer_facenet_full.pt')
test_probs_f=pred3(full_avg_f,test_f); np.save(OUT/'fer_facenet_full_test.npy',test_probs_f)
ensemble_v_f=(1-chosen_f)*ensemble_v_c+chosen_f*valid_probs_f; ensemble_t_f=(1-chosen_f)*ensemble_t_c+chosen_f*test_probs_f
factors_f=(np.bincount(Y,minlength=6)/len(Y))/((np.bincount(ensemble_v_f.argmax(1),minlength=6)+1)/(len(va)+6))
calib_f={str(g):float(accuracy_score(Y[va],(ensemble_v_f*factors_f**g).argmax(1))) for g in [0.,.25,.5,.75,1.]}; gamma_f=max([0.,.25,.5,.75,1.],key=lambda g:calib_f[str(g)])
files_f=[save_submission3(test_probs_f,'submission_fer_facenet.csv'),save_submission3(ensemble_t_f,'submission_fer_five_model_selected.csv'),save_submission3(ensemble_t_f*factors_f**gamma_f,'submission_fer_five_model_calibrated.csv')]
report_f={'date':'2026-10-18','model':'InceptionResnetV1','pretraining':'VGGFace2 identity recognition; facenet-pytorch==2.6.0','pretraining_source':'https://github.com/timesler/facenet-pytorch','normalization':'(pixel - 127.5) / 128','size':160,'seed':seed_f,'best_epoch':best_ep_f,'epochs_trained':len(history_f),'single_validation_accuracy':best_f,'ensemble_validation_scores':scores_f,'selected_face_weight':chosen_f,'calibration_validation_scores':calib_f,'selected_gamma':gamma_f,'full_training_epochs':5,'full_model_validation_accuracy':None,'emotion_training_labels':'official competition training set only','external_test_labels_used':False,'submissions':files_f,'seconds':time.time()-start_f}
(OUT/'fer_face_transfer_20261018.json').write_text(json.dumps(report_f,indent=2)); print('FACE_TRANSFER_FINISHED',json.dumps(report_f),flush=True)


In [ ]:
assert 'report_f' in globals(),'Finish FaceNet training first'
def blend_b(a,b,w,mode):
 if mode=='arithmetic': return (1-w)*a+w*b
 z=(1-w)*np.log(np.clip(a,1e-8,1))+w*np.log(np.clip(b,1e-8,1)); z-=z.max(1,keepdims=True); p=np.exp(z); return p/p.sum(1,keepdims=True)
options_b=[]; candidates_b=[]
for mode_b in ['arithmetic','geometric']:
 specs_b=[]
 for w_b in np.linspace(0,1,11):
  p_b=blend_b(ensemble_v_c,valid_probs_f,float(w_b),mode_b)
  factor_b=(np.bincount(Y,minlength=6)/len(Y))/((np.bincount(p_b.argmax(1),minlength=6)+1)/(len(va)+6))
  for gamma_b in [0.,.25,.5,.75,1.]:
   specs_b.append({'mode':mode_b,'face_weight':float(w_b),'gamma':gamma_b,'validation_accuracy':float(accuracy_score(Y[va],(p_b*factor_b**gamma_b).argmax(1)))})
 choice_b=max(specs_b,key=lambda r:r['validation_accuracy']); options_b+=specs_b
 p_b=blend_b(ensemble_v_c,valid_probs_f,choice_b['face_weight'],mode_b)
 factor_b=(np.bincount(Y,minlength=6)/len(Y))/((np.bincount(p_b.argmax(1),minlength=6)+1)/(len(va)+6))
 t_b=blend_b(ensemble_t_c,test_probs_f,choice_b['face_weight'],mode_b)*factor_b**choice_b['gamma']
 result_b=save_submission3(t_b,'submission_fer_'+mode_b+'_refined.csv'); result_b.update(choice_b); candidates_b.append(result_b)
record_b={'date':'2026-10-18','selection_data':'pre-refit pixel-grouped validation only','description':'bounded arithmetic/geometric probability ensemble search with class-frequency calibration; no test labels','validation_options':options_b,'selected_candidates':candidates_b}
(OUT/'fer_blending_20261018.json').write_text(json.dumps(record_b,indent=2)); print('BLENDING_FINISHED',json.dumps(candidates_b),flush=True)


In [ ]:
# Final reproducibility archive; no training or extra submission.
import importlib.metadata as metadata
for src in OLD.rglob('*'):
    if src.is_file():
        dst=OUT/src.relative_to(OLD)
        if not dst.exists():
            dst.parent.mkdir(parents=True,exist_ok=True)
            shutil.copy2(src,dst)
best_cfg=next(r for r in record_b['selected_candidates'] if r['filename']=='submission_fer_arithmetic_refined.csv')
vp=blend_b(ensemble_v_c,valid_probs_f,best_cfg['face_weight'],'arithmetic')
class_factors=(np.bincount(Y,minlength=6)/len(Y))/((np.bincount(vp.argmax(1),minlength=6)+1)/(len(va)+6))
final_record={
 'competition':'2026task2','date':'2026-10-18',
 'best_public_accuracy':0.741,'previous_public_accuracy':0.717,
 'rank_at_verification':22,'course_grade':None,
 'selected_final_submissions':[
  {'filename':'submission_fer_arithmetic_refined.csv','public_accuracy':0.741},
  {'filename':'submission_fer_geometric_refined.csv','public_accuracy':0.739}],
 'new_public_scores':{'submission_fer_resnet50_calibrated.csv':0.724,'submission_fer_four_model_calibrated.csv':0.726,'submission_fer_five_model_calibrated.csv':0.738,'submission_fer_geometric_refined.csv':0.739,'submission_fer_arithmetic_refined.csv':0.741},
 'best_blend':{'mode':'arithmetic','weights':{'resnet18':0.196875,'efficientnet_b2':0.196875,'resnet50':0.13125,'clahe_resnet34':0.175,'vggface2_inception_resnet_v1':0.3},'class_factors':class_factors.tolist(),'calibration_gamma':0.25,'horizontal_flip_tta':True,'exact_pixel_consistency':True},
 'labels':{'Angry':0,'Fear':1,'Happy':2,'Sad':3,'Surprise':4,'Neutral':5},
 'validation':{'train_rows':22600,'validation_rows':5675,'method':'first split of StratifiedGroupKFold(5, shuffle=True, random_state=42), grouped by pixel hash; not full five-fold cross validation','selection_uses_prefit_validation':True},
 'pretraining':['ImageNet','VGGFace2 face identity pretraining; emotion fine tuning uses competition labels only'],
 'versions':{p:metadata.version(p) for p in ['torch','torchvision','numpy','pandas','scikit-learn','facenet-pytorch']},
 'device':torch.cuda.get_device_name(0),'artifact_files':[]}
manifest_path=OUT/'fer_final_artifact_index_20261018.json'
for p in sorted(OUT.rglob('*')):
    if p.is_file() and p!=manifest_path:
        h=hashlib.sha256()
        with p.open('rb') as handle:
            for block in iter(lambda:handle.read(4*1024*1024),b''):h.update(block)
        final_record['artifact_files'].append({'file':str(p.relative_to(OUT)),'bytes':p.stat().st_size,'sha256':h.hexdigest()})
manifest_path.write_text(json.dumps(final_record,ensure_ascii=False,indent=2))
print('FINAL_ARTIFACTS_PACKED',len(final_record['artifact_files']),sum(x['bytes'] for x in final_record['artifact_files']),'best=',final_record['best_public_accuracy'])
